## Rolling updates & rollback

Deploying a new version is one of the most common Swarm operations, and it happens **without downtime**. The default:

```bash
docker service update --image myorg/api:1.2.4 api
```

Swarm updates **one task at a time**: stop an old task, start a new one, wait for it to be ready, move on. Old and new run side by side during the roll, so the service keeps serving throughout. You tune the rollout with a handful of flags (also settable at `service create` so the policy travels with the service):

```bash
docker service update --image myorg/api:1.2.4 \
  --update-parallelism 2 \
  --update-delay 10s \
  --update-monitor 30s \
  --update-max-failure-ratio 0.2 \
  --update-failure-action rollback \
  api
```

- **`--update-parallelism N`** — update N tasks per batch (default 1).
- **`--update-delay`** — pause between batches.
- **`--update-monitor 30s`** — after starting a task, *watch it* this long before calling it a success — catches "starts fine, crashes after 20s" bugs.
- **`--update-max-failure-ratio 0.2`** — abort if more than 20% of tasks fail their monitor window.
- **`--update-failure-action rollback`** — on abort, automatically revert to the previous image (vs `pause`/`continue`).

That combination is what makes a bad deploy self-correcting: Swarm rolls forward, watches each new task, and if too many fail their monitor window it **rolls itself back** to the last good image — no human in the loop. You can also revert manually any time with `docker service rollback api`. The monitor window is the flag people forget and most need: without it, a container that starts and then crash-loops still counts as "updated successfully."